In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
import gc
import ast
import copy
import time
import random
import numpy as np
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as torchdata

from Model.MENDR.MENDREncoder import MENDREncoder, WaveletEncoderDecoder
from Model.MENDR.MENDRContextualizer import MENDRContextualizer
from Model.MENDR.MENDRTrainer import MENDRTrainer
from Model.MENDR.mAtt.optimizer import MixOptimizer

from dataset import WaveletFinetuningDataset
from torch_geometric.loader import DataLoader
from sklearn.metrics import balanced_accuracy_score
import optuna
from optuna.trial import TrialState


In [3]:
# Load Dataset
print("*" * 50)
train_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/train")
print("Dataset Loaded. Length of Train Dataset: ", len(train_dataset))
eval_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/eval")
print("Dataset Loaded. Length of Validation Dataset: ", len(eval_dataset))


**************************************************
Loading 2130 folders


100%|██████████| 2130/2130 [02:26<00:00, 14.58it/s]


Dataset Loaded. Length of Train Dataset:  44854
Loading 253 folders


100%|██████████| 253/253 [00:13<00:00, 18.52it/s]

Dataset Loaded. Length of Validation Dataset:  5101


In [4]:
### Seed ###
torch.cuda.empty_cache()
random.seed(42)
os.environ['PYTHONHASHSEED'] = str(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
## CUDNN ##
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [18]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
train_loader = DataLoader(train_dataset, batch_size=256, num_workers=2, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=256, num_workers=2, shuffle=False)
print(len(train_loader), len(eval_loader))

176 20


In [19]:
def define_model(trial):
    # We optimize the number of layers, hidden units, dropout ratio in each layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []
    in_features = 1330

    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 95, 380)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(out_features))
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))
        in_features = out_features

    layers.append(nn.Linear(in_features, 2))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)

In [20]:
### Model ###
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = MENDREncoder(device=device)
contextualizer = MENDRContextualizer(device=device)

for param in encoder.parameters():
    param.requires_grad = False

for param in contextualizer.parameters():
    param.requires_grad = False

encoder.load_state_dict(torch.load("checkpoint/batch_size_256_epoch_2/encoder_weights.pth"))
contextualizer.load_state_dict(torch.load("checkpoint/batch_size_256_epoch_2/contextualizer_weights.pth"))

#optim_params = []
#optim_params += list(encoder.parameters())
#optim_params += list(contextualizer.parameters())
#optim_params += list(decoder.parameters())

#optimizer = torch.optim.AdamW(optim_params, lr=0.001)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, threshold=.1, mode='min')
#optimizer = MixOptimizer(optimizer)


<All keys matched successfully>

In [21]:
# Quick hack
contextualizer.wavelet_r2e_lin = contextualizer.wavelet_r2e_lin.to(device)
contextualizer.wavelet_combined_r2e_lin = contextualizer.combined_r2e_lin.to(device)

In [22]:
def _std_norm(x):
	mean = torch.mean(x, dim=(0, 1), keepdim=True)
	std = torch.std(x, dim=(0, 1), keepdim=True)
	x = (x - mean) / std
	return x

In [23]:
def train_one_epoch(encoder, contextualizer, decoder, optimizer, scheduler, task_loss_fn, train_loader):
	loss_sum = 0
	# Freeze foundation weights
	encoder.eval()
	contextualizer.train()
	# Only train decoder
	decoder.train()

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	balanced_accuracies = []
	for iteration in pbar:
		data = next(data_iterator)

		for key, value in data.items():
			if isinstance(value, torch.Tensor):
				# Perform batch normalization across channels 
				data[key] = _std_norm(value.float().to(device))

		relevant_bands = [data['delta'].float().to(device),
		 				  data['theta'].float().to(device),
						  data['alpha'].float().to(device),
						  data['beta'].float().to(device),
						  data['gamma'].float().to(device)]

		inputs = dict(zip(BANDS, relevant_bands))

		with torch.no_grad():
			encoder_outputs = encoder(data['graph'], inputs)
			combined_r2e_output, combined_manifold_output, wavelet_r2e_output, wavelet_manifold_output = contextualizer(encoder_outputs)

		embed_cat = [combined_r2e_output]
		for band, embedding in wavelet_r2e_output.items():
			embed_cat.append(embedding)
		embed_cat = torch.cat(embed_cat, dim=1)

		prediction = decoder(embed_cat)
		ground_truth = data['graph'].y.to(device)

		task_loss = task_loss_fn(prediction, ground_truth)

		#print(torch.argmax(prediction, axis=1).shape)

		y_pred = torch.argmax(prediction, axis=1).cpu().numpy()
		y_true = ground_truth.cpu().numpy()

		correct = (torch.argmax(prediction, axis=1) == ground_truth).sum().item()
		accuracy = correct / prediction.size(0)
		balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
		balanced_accuracies.append(balanced_accuracy)
		loss = task_loss

		scheduler.step(iteration)

		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		loss_sum += loss.item()

		pbar.set_postfix(task_loss=task_loss.item(), accuracy=f'{accuracy*100:.3f}', balanced_accuracy=f'{balanced_accuracy*100:.3f}', lr=scheduler.get_last_lr()[0])
	balanced_accuracies = np.array(balanced_accuracies)	
	print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
	return loss_sum

In [24]:
def validate_one_epoch(encoder, contextualizer, decoder, task_loss_fn, val_loader):
	loss_sum = 0
	encoder.eval()
	contextualizer.eval()
	decoder.eval()

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	balanced_accuracies = []
	with torch.no_grad():
		for iteration in pbar:
			data = next(data_iterator)
			
			for key, value in data.items():
				if isinstance(value, torch.Tensor):
					# Perform batch normalization across channels 
					data[key] = _std_norm(value.float().to(device))

			relevant_bands = [data['delta'].float().to(device),
							data['theta'].float().to(device),
							data['alpha'].float().to(device),
							data['beta'].float().to(device),
							data['gamma'].float().to(device)]

			inputs = dict(zip(BANDS, relevant_bands))

			encoder_outputs = encoder(data['graph'], inputs)
			combined_r2e_output, combined_manifold_output, wavelet_r2e_output, wavelet_manifold_output = contextualizer(encoder_outputs)

			embed_cat = [combined_r2e_output]
			for band, embedding in wavelet_r2e_output.items():
				embed_cat.append(embedding)
			embed_cat = torch.cat(embed_cat, dim=1)

			prediction = decoder(embed_cat)
			ground_truth = data['graph'].y.to(device)

			task_loss = task_loss_fn(prediction, ground_truth)

			loss = task_loss
			loss_sum += loss.item()

			correct = (torch.argmax(prediction, axis=1) == ground_truth).sum().item()
			accuracy = correct / prediction.size(0)

			y_pred = torch.argmax(prediction, axis=1).cpu().numpy()
			y_true = ground_truth.cpu().numpy()
			balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
			balanced_accuracies.append(balanced_accuracy)

			pbar.set_postfix(
				task_loss=task_loss.item(),
				accuracy=f'{accuracy*100:.3f}',
				balanced_accuracy=f'{balanced_accuracy*100:.3f}')

		balanced_accuracies = np.array(balanced_accuracies)
		print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
		return loss_sum, balanced_accuracies.mean(), balanced_accuracies.std()

In [25]:
def objective(trial):
	decoder = define_model(trial).to(device)
	optimizer_name = trial.suggest_categorical("optimizer", ["Adam"])
	lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
	optimizer = getattr(optim, optimizer_name)(decoder.parameters(), lr=lr)
	scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2)
	for epoch_idx in range(1):
		training_loss = train_one_epoch(encoder, contextualizer, decoder, optimizer, scheduler, nn.NLLLoss(), train_loader)
		eval_loss, eval_acc, eval_std = validate_one_epoch(encoder, contextualizer, decoder, nn.NLLLoss(), eval_loader)
	return eval_acc, eval_std
	#	print("Epoch: ", epoch_idx, "Total Training Loss: ", training_loss, "Total Validation Loss:", eval_loss)

In [26]:
study = optuna.create_study(directions=["maximize", "minimize"])
study.optimize(objective, n_trials=15, timeout=6000)

[I 2025-02-11 17:29:45,609] A new study created in memory with name: no-name-0d6064aa-d760-4291-a4f6-d842d107d536
Training: 100%|██████████| 176/176 [10:46<00:00,  3.67s/it, accuracy=75.926, balanced_accuracy=75.105, lr=0.00406, task_loss=0.478]


Mean Acc: 0.7600376124179387 Acc Std: 0.03901647905487791


Validating: 100%|██████████| 20/20 [01:05<00:00,  3.29s/it, accuracy=74.262, balanced_accuracy=83.862, task_loss=0.463]


Mean Acc: 0.752921779014695 Acc Std: 0.11525904119057703


[I 2025-02-11 17:41:39,152] Trial 0 finished with values: [0.752921779014695, 0.11525904119057703] and parameters: {'n_layers': 1, 'n_units_l0': 202, 'dropout_l0': 0.39078328187966715, 'optimizer': 'Adam', 'lr': 0.0058686395841702505}.
Training: 100%|██████████| 176/176 [10:45<00:00,  3.67s/it, accuracy=83.333, balanced_accuracy=83.676, lr=0.000735, task_loss=0.421]


...

[I 2025-02-11 19:05:25,729] Trial 7 finished with values: [0.765624535938438, 0.11548398914262156] and parameters: {'n_layers': 1, 'n_units_l0': 231, 'dropout_l0': 0.34528502980004305, 'optimizer': 'Adam', 'lr': 0.0013333070038566632}.
Training: 100%|██████████| 176/176 [10:53<00:00,  3.71s/it, accuracy=74.074, balanced_accuracy=72.222, lr=5.11e-5, task_loss=0.569]


Mean Acc: 0.6759528565004781 Acc Std: 0.06121826168433074


Validating: 100%|██████████| 20/20 [01:06<00:00,  3.31s/it, accuracy=67.089, balanced_accuracy=79.365, task_loss=0.658]


Mean Acc: 0.755129167957731 Acc Std: 0.11527713201845063


[I 2025-02-11 19:17:26,315] Trial 8 finished with values: [0.755129167957731, 0.11527713201845063] and parameters: {'n_layers': 2, 'n_units_l0': 145, 'dropout_l0': 0.3061558417780449, 'n_units_l1': 347, 'dropout_l1': 0.25315679224234844, 'optimizer': 'Adam', 'lr': 7.388786014883503e-05}.


In [32]:
pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trials = study.best_trials

for trial in trials:
    #print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

Study statistics: 
  Number of finished trials:  9
  Number of pruned trials:  0
  Number of complete trials:  9
Best trial:
  Params: 
    n_layers: 2
    n_units_l0: 164
    dropout_l0: 0.33152213550667764
    n_units_l1: 287
    dropout_l1: 0.3414976509948152
    optimizer: Adam
    lr: 0.0010631408570251214
